# SC-5-Inheritance - Heritage et Interfaces

[<< Functions & State](SC-4-Functions-State.ipynb) | [Errors & Events >>](SC-6-Errors-Events.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'**heritage** simple et multiple
2. Utiliser les **interfaces** pour l'abstraction
3. Maitriser **abstract** et **override**

### Prerequis

- [SC-1](SC-3-Solidity-Basics.ipynb) et [SC-2](SC-4-Functions-State.ipynb) completes
- Bonne maitrise des fonctions et modificateurs Solidity

### Duree estimee : 35 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [1]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity (source a un seul contrat)."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


def deploy_named(w3, source_code, contract_name, deployer, *constructor_args):
    """Compiler une source multi-contrats et deployer un contrat specifique par nom."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    for contract_id, contract_interface in compiled.items():
        if contract_id.split(':')[-1] == contract_name:
            Contract = w3.eth.contract(
                abi=contract_interface["abi"], bytecode=contract_interface["bin"]
            )
            tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
            receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
            instance = w3.eth.contract(
                address=receipt.contractAddress, abi=contract_interface["abi"]
            )
            print(f"Deploye: {contract_name} a {receipt.contractAddress}")
            return instance, receipt
    raise ValueError(f"Contrat '{contract_name}' introuvable dans la source compilee")

Connecte a anvil (chain 31337), deployer: 0xf39Fd6e5...


---

## 1. Heritage simple

Solidity supporte l'heritage avec le mot-cle `is`.

In [2]:
# Heritage simple
INHERITANCE_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Animal {
    string public name;

    constructor(string memory _name) {
        name = _name;
    }

    function speak() public pure virtual returns (string memory) {
        return "...";
    }
}

contract Dog is Animal {
    constructor(string memory _name) Animal(_name) {}

    function speak() public pure override returns (string memory) {
        return "Woof!";
    }
}

contract Cat is Animal {
    constructor(string memory _name) Animal(_name) {}

    function speak() public pure override returns (string memory) {
        return "Meow!";
    }
}
'''

print("--- Heritage simple avec virtual/override ---")
dog, _ = deploy_named(w3, INHERITANCE_BASIC, "Dog", deployer, "Rex")
cat, _ = deploy_named(w3, INHERITANCE_BASIC, "Cat", deployer, "Whiskers")
print(f"  dog.name = '{dog.functions.name().call()}', speak() = '{dog.functions.speak().call()}'")
print(f"  cat.name = '{cat.functions.name().call()}', speak() = '{cat.functions.speak().call()}'")

--- Heritage simple avec virtual/override ---
Deploye: Dog a 0xe7f1725E7734CE288F8367e1Bb143E90bb3F0512


Deploye: Cat a 0xCf7Ed3AccA5a467e9e704C703E8D87F634fB0Fc9
  dog.name = 'Rex', speak() = 'Woof!'
  cat.name = 'Whiskers', speak() = 'Meow!'


### 1.1 Ordre des constructeurs

In [3]:
# Ordre des constructeurs
CONSTRUCTOR_ORDER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract A {
    string public nameA;
    constructor(string memory _name) {
        nameA = _name;
    }
}

contract B is A {
    string public nameB;

    constructor(string memory _nameA, string memory _nameB)
        A(_nameA)
    {
        nameB = _nameB;
    }
}
'''

print("--- Ordre des constructeurs : parent initialise avant enfant ---")
b, _ = deploy_named(w3, CONSTRUCTOR_ORDER, "B", deployer, "Parent", "Child")
print(f"  nameA (depuis A) = '{b.functions.nameA().call()}'")
print(f"  nameB (depuis B) = '{b.functions.nameB().call()}'")

--- Ordre des constructeurs : parent initialise avant enfant ---
Deploye: B a 0x610178dA211FEF7D417bC0e6FeD39F05609AD788
  nameA (depuis A) = 'Parent'
  nameB (depuis B) = 'Child'


---

## 2. Heritage multiple

Solidity permet l'heritage multiple avec resolution de linearisation (C3).

In [4]:
# Heritage multiple
MULTIPLE_INHERITANCE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Ownable {
    address public owner;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}

contract Pausable {
    bool public paused = false;

    modifier whenNotPaused() {
        require(!paused, "Paused");
        _;
    }

    function pause() public {
        paused = true;
    }

    function unpause() public {
        paused = false;
    }
}

contract Token is Ownable, Pausable {
    string public name;
    uint256 public totalSupply;
    mapping(address => uint256) public balances;

    constructor(string memory _name, uint256 _supply) {
        name = _name;
        totalSupply = _supply;
        balances[msg.sender] = _supply;
    }

    function transfer(address to, uint256 amount) public whenNotPaused {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        balances[to] += amount;
    }

    function mint(address to, uint256 amount) public onlyOwner {
        totalSupply += amount;
        balances[to] += amount;
    }
}
'''

print("--- Heritage multiple : Token herite de Ownable et Pausable ---")
token, _ = deploy_named(w3, MULTIPLE_INHERITANCE, "Token", deployer, "MyToken", 1_000_000)
receiver = w3.eth.accounts[1]
print(f"  name = '{token.functions.name().call()}', totalSupply = {token.functions.totalSupply().call()}")
print(f"  deployer balance = {token.functions.balances(deployer).call()}")

# Transfer fonctionne (pas en pause)
token.functions.transfer(receiver, 500).transact({'from': deployer})
print(f"  Apres transfer(500) : receiver balance = {token.functions.balances(receiver).call()}")

# Pause puis transfer bloque
token.functions.pause().transact({'from': deployer})
try:
    token.functions.transfer(receiver, 100).transact({'from': deployer})
    print("  ERREUR : transfer aurait du revert!")
except Exception:
    print(f"  transfer reverte quand paused=True (attendu)")

# Mint via onlyOwner
token.functions.unpause().transact({'from': deployer})
token.functions.mint(receiver, 1000).transact({'from': deployer})
print(f"  Apres mint(1000) : receiver balance = {token.functions.balances(receiver).call()}")

--- Heritage multiple : Token herite de Ownable et Pausable ---


Deploye: Token a 0x0B306BF915C4d645ff596e518fAf3F9669b97016


  name = 'MyToken', totalSupply = 1000000
  deployer balance = 1000000
  Apres transfer(500) : receiver balance = 500
  transfer reverte quand paused=True (attendu)
  Apres mint(1000) : receiver balance = 1500


---

## 3. Interfaces

Les interfaces definissent des signatures sans implementation.

In [5]:
# Interfaces
INTERFACE_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

interface IERC20 {
    function totalSupply() external view returns (uint256);
    function balanceOf(address account) external view returns (uint256);
    function transfer(address to, uint256 amount) external returns (bool);
    function allowance(address owner, address spender) external view returns (uint256);
    function approve(address spender, uint256 amount) external returns (bool);
    function transferFrom(address from, address to, uint256 amount) external returns (bool);

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
}

contract SimpleToken is IERC20 {
    string public constant name = "Simple Token";
    string public constant symbol = "SIM";
    uint8 public constant decimals = 18;

    uint256 private _totalSupply;
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    constructor(uint256 initialSupply) {
        _totalSupply = initialSupply;
        _balances[msg.sender] = initialSupply;
        emit Transfer(address(0), msg.sender, initialSupply);
    }

    function totalSupply() external view override returns (uint256) { return _totalSupply; }
    function balanceOf(address account) external view override returns (uint256) { return _balances[account]; }
    function allowance(address owner, address spender) external view override returns (uint256) { return _allowances[owner][spender]; }

    function transfer(address to, uint256 amount) external override returns (bool) {
        require(_balances[msg.sender] >= amount, "Insufficient balance");
        _balances[msg.sender] -= amount;
        _balances[to] += amount;
        emit Transfer(msg.sender, to, amount);
        return true;
    }

    function approve(address spender, uint256 amount) external override returns (bool) {
        _allowances[msg.sender][spender] = amount;
        emit Approval(msg.sender, spender, amount);
        return true;
    }

    function transferFrom(address from, address to, uint256 amount) external override returns (bool) {
        require(_allowances[from][msg.sender] >= amount, "Insufficient allowance");
        require(_balances[from] >= amount, "Insufficient balance");
        _allowances[from][msg.sender] -= amount;
        _balances[from] -= amount;
        _balances[to] += amount;
        emit Transfer(from, to, amount);
        return true;
    }
}
'''

print("--- Interface IERC20 implementee par SimpleToken ---")
erc20, _ = deploy_named(w3, INTERFACE_EXAMPLE, "SimpleToken", deployer, 1_000_000)
receiver = w3.eth.accounts[1]
spender = w3.eth.accounts[2]
print(f"  totalSupply() = {erc20.functions.totalSupply().call()}")
print(f"  balanceOf(deployer) = {erc20.functions.balanceOf(deployer).call()}")

# Transfer
erc20.functions.transfer(receiver, 1000).transact({'from': deployer})
print(f"  Apres transfer(1000) : receiver balance = {erc20.functions.balanceOf(receiver).call()}")

# Approve + transferFrom
erc20.functions.approve(spender, 500).transact({'from': deployer})
print(f"  allowance(deployer, spender) = {erc20.functions.allowance(deployer, spender).call()}")
erc20.functions.transferFrom(deployer, receiver, 200).transact({'from': spender})
print(f"  Apres transferFrom(200) : receiver balance = {erc20.functions.balanceOf(receiver).call()}")

--- Interface IERC20 implementee par SimpleToken ---


Deploye: SimpleToken a 0x9E545E3C0baAB3E08CdfD552C960A1050f373042
  totalSupply() = 1000000
  balanceOf(deployer) = 1000000


  Apres transfer(1000) : receiver balance = 1000
  allowance(deployer, spender) = 500
  Apres transferFrom(200) : receiver balance = 1200


---

## 4. Contrats abstraits

Les contrats abstraits peuvent avoir des fonctions implementees ET non implementees.

In [6]:
# Contrats abstraits
ABSTRACT_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

abstract contract PriceFeed {
    function getPrice() public view virtual returns (uint256);

    function getPriceWithDecimals() public view virtual returns (uint256) {
        return getPrice() * 10 ** 18;
    }
}

contract ChainlinkPriceFeed is PriceFeed {
    uint256 private price;

    constructor(uint256 _price) {
        price = _price;
    }

    function getPrice() public view override returns (uint256) {
        return price;
    }
}

contract UniswapPriceFeed is PriceFeed {
    uint256 private price;

    constructor(uint256 _price) {
        price = _price;
    }

    function getPrice() public view override returns (uint256) {
        return price;
    }

    function getPriceWithDecimals() public view override returns (uint256) {
        return getPrice() * 10 ** 6;
    }
}
'''

print("--- Contrats abstraits : implementations differentes ---")
chainlink, _ = deploy_named(w3, ABSTRACT_EXAMPLE, "ChainlinkPriceFeed", deployer, 100)
uniswap, _ = deploy_named(w3, ABSTRACT_EXAMPLE, "UniswapPriceFeed", deployer, 200)

print(f"  Chainlink getPrice() = {chainlink.functions.getPrice().call()}")
print(f"  Chainlink getPriceWithDecimals() = {chainlink.functions.getPriceWithDecimals().call()} (x10^18)")
print(f"  Uniswap  getPrice() = {uniswap.functions.getPrice().call()}")
print(f"  Uniswap  getPriceWithDecimals() = {uniswap.functions.getPriceWithDecimals().call()} (x10^6, override)")

--- Contrats abstraits : implementations differentes ---
Deploye: ChainlinkPriceFeed a 0x99bbA657f2BbC93c02D617f8bA121cB8Fc104Acf


Deploye: UniswapPriceFeed a 0x5eb3Bc0a489C5A8288765d2336659EbCA68FCd00
  Chainlink getPrice() = 100
  Chainlink getPriceWithDecimals() = 100000000000000000000 (x10^18)
  Uniswap  getPrice() = 200
  Uniswap  getPriceWithDecimals() = 200000000 (x10^6, override)


---

## 5. Super et linearisation

Le mot-cle `super` appelle la fonction du contrat parent.

In [7]:
# Super et linearisation
SUPER_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract A {
    function foo() public pure virtual returns (string memory) {
        return "A";
    }
}

contract B is A {
    function foo() public pure virtual override returns (string memory) {
        return string.concat(super.foo(), " B");
    }
}

contract C is A {
    function foo() public pure virtual override returns (string memory) {
        return string.concat(super.foo(), " C");
    }
}

// Linearisation C3 : D -> B -> C -> A
// super dans B appelle C (pas A directement)
contract D is B, C {
    function foo() public pure override(B, C) returns (string memory) {
        return string.concat(super.foo(), " D");
    }
}
'''

print("--- Linearisation C3 et super ---")
d_contract, _ = deploy_named(w3, SUPER_EXAMPLE, "D", deployer)
result = d_contract.functions.foo().call()
print(f"  D.foo() = '{result}'")
print(f"  Ordre de resolution : A -> C -> B -> D => '{result}'")

--- Linearisation C3 et super ---
Deploye: D a 0xb7278A61aa25c888815aFC32Ad3cC52fF24fE575
  D.foo() = 'A B C D'
  Ordre de resolution : A -> C -> B -> D => 'A B C D'


---

## 6. Exercices

### Exercice 1 : Contrat Vault

Creez un contrat Vault qui herite de Ownable et ajoute une fonction de depot/withdraw.

In [8]:
# Exercice 1 : Contrat Vault
EXERCICE_VAULT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract Ownable {
    address public owner;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}

contract Vault is Ownable {
    mapping(address => uint256) public balances;

    function deposit() public payable {
        // TODO: Crediter le solde de msg.sender avec msg.value
    }

    function withdraw(uint256 amount) public {
        // TODO: Verifier que l utilisateur a assez de fonds
        // TODO: Reduire le solde
        // TODO: Transferer les ETH
    }

    function withdrawAll() public onlyOwner {
        // TODO: Transferer tout le solde du contrat au proprietaire
    }
}
'''

# Compilation et deploiement reel sur anvil
ownable, receipt = compile_and_deploy(w3, EXERCICE_VAULT, deployer)

Deploye: Vault a 0x82e01223d51Eb87e16A03E24687EDF0F294da6f1

### Exercice 2 : Interface IERC721 simplifiee

Creez une interface NFT simplifiee avec transfer et ownerOf.

In [9]:
# Exercice 2 : Interface IERC721 simplifiee
EXERCICE_NFT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

interface IERC721 {
    function ownerOf(uint256 tokenId) external view returns (address);
    function transferFrom(address from, address to, uint256 tokenId) external;
    event Transfer(address indexed from, address indexed to, uint256 indexed tokenId);
}

contract SimpleNFT is IERC721 {
    mapping(uint256 => address) private _owners;
    uint256 private _nextTokenId = 1;

    function mint() public returns (uint256) {
        // TODO: Generer un nouveau tokenId a partir de _nextTokenId
        // TODO: Assigner le token a msg.sender dans _owners
        // TODO: Emettre l evenement Transfer (from address(0))
        // TODO: Retourner le tokenId
    }

    function ownerOf(uint256 tokenId) public view override returns (address) {
        // TODO: Verifier que le token existe (owner != address(0))
        // TODO: Retourner le proprietaire
    }

    function transferFrom(address from, address to, uint256 tokenId) public override {
        // TODO: Verifier que from est bien le proprietaire
        // TODO: Changer le proprietaire dans _owners
        // TODO: Emettre l evenement Transfer
    }
}
'''

# Compilation et deploiement reel sur anvil
simplenft, receipt = compile_and_deploy(w3, EXERCICE_NFT, deployer)

Deploye: SimpleNFT a 0x7bc06c482DEAd17c0e297aFbC32f6e63d3846650


---

## 7. Resume

| Concept | Description |
|---------|-------------|
| `is` | Heritage d'un contrat parent |
| `virtual` | Fonction pouvant etre overidee |
| `override` | Redefinition d'une fonction parente |
| `interface` | Contrat 100% abstrait |
| `abstract` | Contrat partiellement implemente |
| `super` | Appel a la fonction parente |

---

**Notebook suivant** : [SC-6-Errors-Events](SC-6-Errors-Events.ipynb)

---

[<< Functions & State](SC-4-Functions-State.ipynb) | [Errors & Events >>](SC-6-Errors-Events.ipynb)